<a href="https://colab.research.google.com/github/YoussefAli88/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [16]:
!git clone https://github.com/YoussefAli88/flyrank-ml-internship-starter.git

fatal: destination path 'flyrank-ml-internship-starter' already exists and is not an empty directory.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item, for one month.

Each row represents a single content item's (page's) Search Console performance over a month (month=2026-03), built by aggregating the daily grain of fact_content_daily_performance (one row per report_date, client_hash_id, content_hash_id) up to a monthly grain via GROUP BY content item + month. One row being one content item; not a whole client/website is necessary because two content items belonging to the same client can have very different CTR/position performance, and Lane 4's recommendations are made per content item, not per client. The usage of Monthly updates (rather than daily) is used because click/impression volume per content item is often low; shorter periods of time would make CTR too noisy to support a right flagging decision. Only rows where gsc_data_available = TRUE are used, since GSC data may not yet be fully gathered for a given report date, and unavailable rows shouldn't count toward a monthly aggregate.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Label: 'trend_pct' the target this task scores/predicts.

Features (5 max), all from dim_content: search_volume, competition, content_type, main_intent, word_count. Each of these is known independently of a content item's actual click/impression performance. None require having already known the item's GSC clicks, impressions, or position for the month.

Context: content_hash_id (identifies which content item this row belongs to) and the derived month (from report_date). These identify which row this is, which content item, which time period, but are not used as predictive signal. Including a raw identifier as a feature would let the model memorize individual content items rather than generalize.

Excluded: gsc_clicks, gsc_impressions, gsc_avg_position (and any CTR derived from them), plus any trend-direction column. All are excluded for the same reason: they were used to construct the label (trend_pct) itself. Including them as features would let the model reconstruct the label formula instead of learning a genuine pattern. Which is considered 'Leakage' as we just knew this term in past assignments.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [17]:
!pip install duckdb --quiet

In [18]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

base = "hf://datasets/FlyRank/internship-warehouse"

The first query confirms the fact_content_daily_performance table's natural grain is daily: content items show up to 31 rows in March 2026 (one per day), not one. The second query re-runs the same duplicate-check after aggregating to content item + month, and returns 0 rows; confirming that once aggregated, the claimed unit of analysis ("one row = one content item, one month") holds with no duplicates.

In [19]:
q1_grain = con.sql("""
    SELECT
        content_hash_id,
        DATE_TRUNC('month', report_date) AS month,
        COUNT(*) AS row_count
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available = TRUE
    GROUP BY content_hash_id, DATE_TRUNC('month', report_date)
    HAVING COUNT(*) > 1
    LIMIT 5
""")
q1_grain

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────────┬────────────┬───────────┐
│     content_hash_id      │   month    │ row_count │
│         varchar          │    date    │   int64   │
├──────────────────────────┼────────────┼───────────┤
│ content_b7e512995f79d5a6 │ 2026-03-01 │        31 │
│ content_a7da352b73b02668 │ 2026-03-01 │        31 │
│ content_d056587ff7faca0c │ 2026-03-01 │        31 │
│ content_bfd1e41c2af250c8 │ 2026-03-01 │        21 │
│ content_2662845f598544ef │ 2026-03-01 │        30 │
└──────────────────────────┴────────────┴───────────┘

In [20]:
q1_grain_after_agg = con.sql("""
    SELECT content_hash_id, month, COUNT(*) AS c
    FROM (
        SELECT
            content_hash_id,
            DATE_TRUNC('month', report_date) AS month
        FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available = TRUE
        GROUP BY content_hash_id, DATE_TRUNC('month', report_date)
    )
    GROUP BY content_hash_id, month
    HAVING c > 1
""")
q1_grain_after_agg

┌─────────────────┬───────┬───────┐
│ content_hash_id │ month │   c   │
│     varchar     │ date  │ int64 │
├─────────────────┴───────┴───────┤
│             0 rows              │
└─────────────────────────────────┘

Query 2: After filtering to gsc_data_available = TRUE, the month=2026-03 partition contains 3,611,061 rows spanning the full calendar month (2026-03-01 to 2026-03-31). This measures that the slice covers the entire target month with no missing chunks of dates at the partition level. But note: this MIN/MAX check is computed across the whole partition, not per content item, so individual content items may still have partial-month coverage if the client started mid-month (Would mention that in section 4 -Data Limits-).

In [21]:
q2_count_span = con.sql("""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS earliest_date,
        MAX(report_date) AS latest_date
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available = TRUE
""")
q2_count_span

┌───────────┬───────────────┬─────────────┐
│ row_count │ earliest_date │ latest_date │
│   int64   │     date      │    date     │
├───────────┼───────────────┼─────────────┤
│   3611061 │ 2026-03-01    │ 2026-03-31  │
└───────────┴───────────────┴─────────────┘

Query 3 : Of the 9,841,378 raw rows in the month=2026-03 partition, only 3,611,061 rows (≈37%) have gsc_data_available = TRUE. The remaining ≈63% are dropped by this filter. This is not a data quality problem, as a share of clients either lack GSC access entirely (access_profile values like no_search_or_analytics_access) or have GSC history starting after the analysis window (dim_clients.gsc_data_start), both of which produce gsc_data_available = FALSE.

In [22]:
q3_availability = con.sql("""
    SELECT
        (SELECT COUNT(*) FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')) AS total_rows,
        COUNT(*) AS available_rows
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available = TRUE
""")
q3_availability

┌────────────┬────────────────┐
│ total_rows │ available_rows │
│   int64    │     int64      │
├────────────┼────────────────┤
│    9841378 │        3611061 │
└────────────┴────────────────┘

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice cannot tell you what search performance looked like before a client's GSC access began, or during periods with no GSC access at all. History depth differs per client. Some clients' GSC data starts partway through (or entirely after) the analysis window, and some clients have access_profile = no_search_or_analytics_access and contribute no usable GSC rows at any point. This is why only ≈37% of raw month=2026-03 rows survive the gsc_data_available = TRUE filter. The usable slice for Lane 4 scoring is a lot smaller than the raw table, and is not evenly distributed across clients.

A few other related limitations worth naming:

1- Partial-month coverage is hidden at the aggregate level. Query 2 observed that the slice spans the full calendar month overall, but that MIN/MAX check is computed across the whole partition; however, individual content items may still have fewer than 31 days of underlying data if their client's history started mid-month.
2- This data cannot support causal claims. It can show observed, directional patterns (example: which content items are visible-but-underperforming in a given month), but cannot make sure that a title/meta rewrite caused any change in performance,that would require an experimental design (example: before/after comparison).
3- The final month (June 2026) is intentionally excluded from all iteration and label logic in this contract. It is reserved as a sealed test month, so nothing in this notebook was developed or verified against it.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.